In [ ]:
# NOTEBOOK 33 - regional upwind stations around Delhi: load the CPCB files you downloaded and align them to the Delhi hourly grid
# Input : D:\\FYP\\Eco-Twin\\Raw Sets Regional\\Raw_Data_<year>_site_<id>_<name>_1Hr.csv   (read only; the files are never changed)
# Output: v2_clean/regional_cache/regional_dataset.npz  (new folder, one new file) with, for R regional stations and the 19,921 Delhi hours:
#         PM2.5 (with NaN where the station did not report), a 'reported' mask, ERA5 wind at each station, coordinates, distance and bearing from Delhi.
# Missing hours are NOT filled in: the Feb-Sep 2025 gap in Haryana stays missing and is flagged, so later models know where regional data is absent.
# Also prints diagnostics, including whether regional PM2.5 LEADS Delhi PM2.5 (the signal a lagged upwind feature would use).
# HOW TO CHECK: this cell only imports and sets paths.
import os, re, glob, json, time
import numpy as np, pandas as pd, requests
V2_DIR = r'D:\FYP\Eco-Twin\v2_clean'
RAW_DIR = r'D:\FYP\Eco-Twin\Raw Sets Regional'
OUT_DIR = os.path.join(V2_DIR, 'regional_cache'); os.makedirs(OUT_DIR, exist_ok=True)
ds = pd.read_csv(os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv'), parse_dates=['Timestamp'])
GRID = pd.DatetimeIndex(sorted(ds['Timestamp'].unique())); T = len(GRID)
delhi_pm = ds.pivot_table(index='Timestamp', columns='Station', values='pm2.5').reindex(GRID)
DELHI_MEAN = delhi_pm.mean(axis=1).values                                   # city-mean PM2.5 per hour (the Delhi dataset is gap-free)
DELHI = (28.64, 77.17)
print('Delhi grid:', T, 'hours', GRID[0], '->', GRID[-1])

In [ ]:
# 1) read every station: one row per hour, PM2.5 plus the measured wind where it exists
files = sorted(glob.glob(os.path.join(RAW_DIR, 'Raw_Data_*_1Hr.csv')))
sites = {}
for f in files:
    m = re.match(r'Raw_Data_(\d{4})_site_(\d+)_(.+)_1Hr\.csv', os.path.basename(f))
    sites.setdefault((m.group(2), m.group(3)), []).append(f)
print(len(sites), 'stations,', len(files), 'files')

def norm(s): return re.sub(r'[^a-z0-9]+', '', s.lower())          # drop spaces, commas, hyphens and underscores so 'Sector-12, Karnal' matches 'sector12_karnal'
locs = json.load(open(os.path.join(V2_DIR, 'crosscity_cache', 'india_locations.json')))
by_name = {norm(L['name']): L for L in locs if (L.get('coordinates') or {}).get('latitude') is not None}
R_IDS, R_NAMES, R_LAT, R_LON, PM, WS, WD = [], [], [], [], [], [], []
for (sid, name), fl in sorted(sites.items(), key=lambda t: t[0][1]):
    L = by_name.get(norm(name))
    if L is None:                                                          # tolerate the portal and OpenAQ spelling slightly differently
        cand = [k for k in by_name if k.startswith(norm(name)[:16])]
        L = by_name[cand[0]] if cand else None
    assert L is not None, f'no coordinates found for {name} - tell me and I will add them by hand'
    d = pd.concat([pd.read_csv(f, na_values=['NA', '', 'None']) for f in fl], ignore_index=True)
    d['Timestamp'] = pd.to_datetime(d['Timestamp']); d = d.drop_duplicates('Timestamp').set_index('Timestamp').reindex(GRID)
    pm = d[[c for c in d.columns if c.startswith('PM2.5')][0]].astype(float); pm = pm.where((pm >= 0) & (pm < 1000))        # 1000 is the portal ceiling
    ws = d[[c for c in d.columns if c.startswith('WS')][0]].astype(float); wd = d[[c for c in d.columns if c.startswith('WD')][0]].astype(float)
    R_IDS.append(sid); R_NAMES.append(L['name']); R_LAT.append(L['coordinates']['latitude']); R_LON.append(L['coordinates']['longitude'])
    PM.append(pm.values); WS.append(ws.values); WD.append(wd.values)
PM = np.stack(PM, axis=1).astype(np.float32); WS = np.stack(WS, axis=1).astype(np.float32); WD = np.stack(WD, axis=1).astype(np.float32)
R = PM.shape[1]; REPORTED = ~np.isnan(PM)
R_LAT, R_LON = np.array(R_LAT), np.array(R_LON)

def km(a, b, c, d_):
    p1, p2 = np.radians(a), np.radians(c); x = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(d_ - b) / 2) ** 2
    return 12742 * np.arcsin(np.sqrt(x))
def bearing(lat1, lon1, lat2, lon2):
    p1, p2, dl = np.radians(lat1), np.radians(lat2), np.radians(lon2 - lon1)
    return (np.degrees(np.arctan2(np.sin(dl) * np.cos(p2), np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl))) + 360) % 360
DIST = km(DELHI[0], DELHI[1], R_LAT, R_LON); BEAR = bearing(DELHI[0], DELHI[1], R_LAT, R_LON)
print(f"{'station':44}{'km':>6}{'bearing':>9}{'PM2.5 %':>9}{'wind %':>8}")
for i in range(R):
    print(f'{R_NAMES[i][:44]:44}{DIST[i]:>6.0f}{BEAR[i]:>9.0f}{100 * REPORTED[:, i].mean():>9.0f}{100 * (~np.isnan(WS[:, i]) & ~np.isnan(WD[:, i])).mean():>8.0f}')

# HOW TO CHECK: 8 stations, distances 24-250 km, bearings in the west to north range (about 267-352 deg).

In [ ]:
# 2) ERA5 wind at each regional station (same source and u/v convention as the Delhi stations); cached
def fetch(lat, lon, retries=4):
    p = {'latitude': lat, 'longitude': lon, 'start_date': GRID[0].strftime('%Y-%m-%d'), 'end_date': GRID[-1].strftime('%Y-%m-%d'),
         'hourly': 'wind_speed_10m,wind_direction_10m', 'wind_speed_unit': 'ms', 'timezone': 'Asia/Kolkata'}
    for k in range(retries):
        r = requests.get('https://archive-api.open-meteo.com/v1/archive', params=p, timeout=120)
        if r.status_code == 200:
            h = r.json()['hourly']; return pd.DataFrame({'ws': h['wind_speed_10m'], 'wd': h['wind_direction_10m']}, index=pd.DatetimeIndex(h['time'])).reindex(GRID)
        time.sleep(8 * (k + 1))
    raise RuntimeError('Open-Meteo failed: ' + r.text[:200])
UV = np.zeros((T, R, 2), dtype=np.float32)
for i in range(R):
    d = fetch(R_LAT[i], R_LON[i]); ws, wd = d['ws'].values.astype(float), np.deg2rad(d['wd'].values.astype(float) % 360)
    UV[:, i, 0], UV[:, i, 1] = np.nan_to_num(-ws * np.sin(wd)), np.nan_to_num(-ws * np.cos(wd)); time.sleep(1.0)
print('ERA5 wind at the regional stations: mean speed', float(np.hypot(UV[..., 0], UV[..., 1]).mean()), 'm/s')
np.savez_compressed(os.path.join(OUT_DIR, 'regional_dataset.npz'), pm=PM, reported=REPORTED, era5=UV, ws=WS, wd=WD, lat=R_LAT, lon=R_LON,
                    names=np.array(R_NAMES, dtype=str), dist_km=DIST, bearing=BEAR, grid=np.array(GRID.strftime('%Y-%m-%d %H:%M:%S'), dtype='U19'))
print('saved', os.path.join(OUT_DIR, 'regional_dataset.npz'))

In [ ]:
# 3) how much regional data will the Delhi windows have?  (a window = 48 input hours + 24 forecast hours)
n_rep = REPORTED.sum(axis=1)                                                  # stations reporting in each hour
months = pd.Series(n_rep, index=GRID).groupby(GRID.to_period('M')).mean().round(1)
print('average number of regional stations reporting per hour, by month (of', R, '):')
print(months.to_string())
val_end, tr_end = int(T * 0.85), int(T * 0.70)
def share(lo, hi, need=4):                                                     # share of hours with at least `need` stations reporting
    return float((n_rep[lo:hi] >= need).mean())
print()
print(f'hours with at least 4 of {R} regional stations reporting:  train {100 * share(0, tr_end):.0f}%  |  validation {100 * share(tr_end, val_end):.0f}%  |  test {100 * share(val_end, T):.0f}%')
print(f'hours with at least 1 regional station reporting:          train {100 * share(0, tr_end, 1):.0f}%  |  validation {100 * share(tr_end, val_end, 1):.0f}%  |  test {100 * share(val_end, T, 1):.0f}%')

# HOW TO CHECK: the test share is high (about 90%+); the training share is lower because of the Feb-Sep 2025 Haryana gap.

In [ ]:
# 4) does regional PM2.5 LEAD Delhi PM2.5?  correlation of log PM2.5 at the regional station (lagged k hours) with the Delhi city mean now
winter = np.isin(GRID.month, [10, 11, 12, 1, 2, 3])
ld = np.log1p(DELHI_MEAN)
print('station                                      km  bearing | corr at lag 0 | best lag (h) | corr at best lag | corr at 6 h lag')
rows = []
for i in range(R):
    lr = np.log1p(PM[:, i]); best = (-9, 0.0); c0 = np.nan; c6 = np.nan
    for k in range(0, 25):
        x = lr[:T - k] if k else lr; y = ld[k:] if k else ld; w = winter[k:] if k else winter
        ok = np.isfinite(x) & np.isfinite(y) & w
        if ok.sum() < 1000: continue
        c = float(np.corrcoef(x[ok], y[ok])[0, 1])
        if k == 0: c0 = c
        if k == 6: c6 = c
        if c > best[1] or best[0] < 0: best = (k, c) if (best[0] < 0 or c > best[1]) else best
    rows.append((R_NAMES[i], DIST[i], BEAR[i], c0, best[0], best[1], c6))
for r in rows: print(f'{r[0][:42]:42}{r[1]:>5.0f}{r[2]:>8.0f} | {r[3]:>13.3f} | {r[4]:>12d} | {r[5]:>16.3f} | {r[6]:>15.3f}')
print()
print('Reading: a station "leads" Delhi if its correlation is highest at a positive lag; a peak at 0 means it moves with Delhi at the same hour.')
print('Winter months only (Oct-Mar), hours where both exist.')

# HOW TO READ: peak lags of roughly 2-10 hours at stations 80-200 km away would support a lagged upwind feature. All peaks at 0 means the regional stations
# mostly track the same synoptic weather as Delhi and a lag will add little beyond a same-hour regional mean.